# Sinh Caption cho Data Augmentation với LLaVA
Notebook này duyệt qua các thư mục `train`, `validation`, `test` trong thư mục data đã aug (`PlantDocSplited_depth_AUG`) và sử dụng LLaVA để sinh mô tả hình thái chi tiết cho từng ảnh.

In [1]:
import os
import json
import torch
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
from transformers import AutoProcessor, LlavaForConditionalGeneration, BitsAndBytesConfig

In [2]:
# =========================
# 1. CẤU HÌNH ĐƯỜNG DẪN & MÔI TRƯỜNG
# =========================
current_dir = Path.cwd()
PROJECT_ROOT = current_dir
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
print(f"Project Root: {PROJECT_ROOT}")

INPUT_BASE = PROJECT_ROOT / "data" / "processed" / "PlantDocSplited_depth_AUG"
OUTPUT_BASE = PROJECT_ROOT / "data" / "processed" / "captions_LLaVA_depth_AUG"

# CỜ RESET: Đặt True nếu muốn xóa sạch caption cũ để chạy lại từ đầu với prompt mới.
# LƯU Ý: Nếu bị ngắt kết nối giữa chừng, trước khi chạy lại notebook, bạn PHẢI đổi 
# biến này thành False để không bị xóa mất những ảnh vừa mới gen xong.
START_FRESH = True
if START_FRESH:
    import shutil
    if OUTPUT_BASE.exists():
        shutil.rmtree(OUTPUT_BASE)
        print("Đã xóa toàn bộ thư mục caption cũ. Sẵn sàng sinh lại từ đầu!")

# Các thư mục con cần sinh caption
SPLITS = ["train", "validation", "test"]

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device: {DEVICE}")

Project Root: /media/data3/users/luongdth/MulCo-PlantNet
Using Device: cuda


In [3]:
# =========================
# 2. PROMPT TEMPLATE
# =========================
PROMPT_TEMPLATE = """
Act as an expert plant pathologist conducting a detailed visual inspection. Provide a comprehensive, structured description of the leaf's condition, focusing exclusively on pathological and physiological features. 

CRITICAL RULES: 
- DO NOT state, guess, or imply the name of the plant species (e.g., do not use words like Tomato, Apple, Corn, Potato).
- DO NOT name the specific disease or pathogen (e.g., do not say Early Blight, Rust, Mosaic Virus, Scab). 
- DO NOT provide a final diagnosis. 
- Restrict your output STRICTLY to observable visual symptoms. Do not describe the background, lighting, or irrelevant objects.

If the leaf appears completely healthy:
Describe its healthy state in detail. Note the uniform coloration, intact structural integrity, natural texture, and the explicit absence of any lesions, discoloration, pest damage, fungal growth, or viral deformations.

If the leaf exhibits signs of disease, pathogens, or pest damage, systematically describe the symptoms using the following aspects:
1. Color & Pigmentation: Describe any abnormal discoloration, including general chlorosis (yellowing), distinct mosaic/mottling patterns, or specific color changes in affected areas.
2. Structural Deformation: Note any physical distortions such as leaf curling, wrinkling, stunting, or wilting.
3. Spot & Lesion Morphology: Detail the characteristics of any spots or lesions—specify their color, shape (e.g., angular, circular, irregular), internal patterns (e.g., target-like concentric rings, water-soaked appearance), and whether they have distinct borders or chlorotic halos.
4. Pathogen & Pest Signs: Report any visible evidence of the causal agent, such as fungal fuzzy mold, powdery mildew, rust pustules, or pest indicators like spider mite stippling, webbing, or insect feeding holes.
5. Tissue Necrosis & Distribution: Describe the extent of dead tissue (necrosis), blighting, structural collapse, and how these symptoms are distributed across the leaf (e.g., at the margins, interveinal, or randomly scattered).

Provide the final output as a cohesive, professional pathological report in a single well-connected paragraph. Maximize the use of precise botanical and pathological terminology without violating the critical rules.
"""

In [4]:
# =========================
# 3. TẢI MÔ HÌNH LLaVA
# =========================
print(f"Đang tải LLaVA model {MODEL_ID} với cấu hình 4-bit quantization...")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID, 
    quantization_config=quantization_config, 
    device_map="auto",
    low_cpu_mem_usage=True
)
print("Tải mô hình thành công!")

Đang tải LLaVA model llava-hf/llava-1.5-7b-hf với cấu hình 4-bit quantization...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

Tải mô hình thành công!


In [5]:
# =========================
# 4. VÒNG LẶP SINH CAPTION
# =========================
for split in SPLITS:
    input_dir = INPUT_BASE / split
    output_dir = OUTPUT_BASE / split
    
    if not input_dir.exists():
        print(f"[Cảnh báo] Không tìm thấy thư mục: {input_dir}, Bỏ qua...")
        continue
        
    output_dir.mkdir(parents=True, exist_ok=True)
    class_dirs = sorted([d for d in input_dir.iterdir() if d.is_dir()])
    
    print(f"\n=========================================")
    print(f"Bắt đầu xử lý tập dữ liệu: {split.upper()}")
    print(f"=========================================")
    
    for idx, class_dir in enumerate(class_dirs):
        class_name = class_dir.name
        out_json_path = output_dir / f"{class_name}.json"
        
        results = {}
        # Hỗ trợ resume nếu bị ngắt quãng giữa chừng
        if out_json_path.exists():
            with open(out_json_path, "r", encoding="utf-8") as f:
                results = json.load(f)
                
        image_paths = sorted([p for p in class_dir.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
        
        print(f"  Class [{idx+1}/{len(class_dirs)}]: {class_name} | Có {len(image_paths)} ảnh")
        
        for img_path in tqdm(image_paths, desc=class_name, leave=False):
            img_name = img_path.name
            if img_name in results:
                continue
                
            try:
                image = Image.open(img_path).convert("RGB")
            except Exception as e:
                print(f"Lỗi khi đọc ảnh {img_name}: {e}")
                continue
            
            chat_prompt = f"USER: <image>\n{PROMPT_TEMPLATE}\nASSISTANT:"
            inputs = processor(text=chat_prompt, images=image, return_tensors="pt").to(DEVICE, torch.float16)
            
            with torch.no_grad():
                generate_ids = model.generate(
                    **inputs, 
                    max_new_tokens=128, 
                    temperature=0.2, 
                    do_sample=True
                )
            
            generated_text = processor.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
            response = generated_text.split("ASSISTANT:")[-1].strip() if "ASSISTANT:" in generated_text else generated_text.strip()
                
            results[img_name] = {"text": response, "label": idx}
            
            # Lưu dữ liệu ngay sau mỗi ảnh để đảm bảo an toàn tuyệt đối, có thể resume nếu bị ngắt kết nối
            with open(out_json_path, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)

print("\nHoàn tất sinh caption cho toàn bộ dữ liệu!")


Bắt đầu xử lý tập dữ liệu: TRAIN
  Class [1/28]: Apple_Scab_Leaf | Có 146 ảnh


Apple_Scab_Leaf:   0%|          | 0/146 [00:00<?, ?it/s]

  Class [2/28]: Apple_leaf | Có 138 ảnh


Apple_leaf:   0%|          | 0/138 [00:00<?, ?it/s]

  Class [3/28]: Apple_rust_leaf | Có 168 ảnh


...

  Class [27/28]: grape_leaf | Có 12 ảnh


grape_leaf:   0%|          | 0/12 [00:00<?, ?it/s]

  Class [28/28]: grape_leaf_black_rot | Có 8 ảnh


grape_leaf_black_rot:   0%|          | 0/8 [00:00<?, ?it/s]


Hoàn tất sinh caption cho toàn bộ dữ liệu!
